In [1]:
from src.missing_data_generator import MissingDataGenerator
from src.gaussian_mixture import GaussianMixtureMAR
import numpy as np
import torch
from src.metrics import energy_distance_faster

from joblib import Parallel, delayed
import matplotlib.pyplot as plt

Error importing in API mode: ImportError("dlopen(/Users/huimin/Documents/UNIGE PhD/research/nonparametric/.venv/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <1984CB9B-890E-3855-A246-270AC6AC4C62> /Users/huimin/Documents/UNIGE PhD/research/nonparametric/.venv/lib/python3.11/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


In [2]:
n = 2000
d = 20
n_pat = 3
k_range = (0,20)
eps_vals = [0.0, 0.1, 0.2, 0.5, 0.7, 0.9]
B = 2

In [3]:
mdg = MissingDataGenerator(n=n, d=d, n_pat=n_pat, seed=1, distr='logistic') # the patterns is sampled once in the init

def job(
        k_range:tuple = None,
        eps:int = None
):
    X = mdg.get_X()
    _, M = mdg.get_M_ampute_eps(X, eps=eps)
    Xtest = mdg.get_X()
    model = GaussianMixtureMAR(k_range=k_range)
    model.fit(X, M)
    Xtilde = model.sample(n)[0]
    return energy_distance_faster(Xtest, Xtilde, scale=True).item()

In [4]:
B = 2

In [ ]:
results = Parallel(n_jobs=15)(
    delayed(job)(k_range=k_range, eps=eps)
    for _ in range(B)
    for eps in eps_vals
)

results = np.asarray(results).reshape(B, -1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(results, tick_labels=eps_vals, patch_artist=True)

ax.set_ylabel("Scaled Energy distance", fontsize=12)
ax.set_xlabel(r"$\epsilon$", fontsize=12)

ax.tick_params(axis="both", labelsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.5)

fig.tight_layout()

plt.savefig(f'results/density_estimation_{n_pat}_patterns_{B}_B.png',dpi=300,
    bbox_inches="tight",)

plt.show()